In [1]:
!pip install pyannote.audio

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 2.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.6/59.6 kB 2.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 893.7/893.7 kB 31.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 853.6/853.6 kB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 72.1/72.1 kB 5.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.0/142.0 kB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.7/68.7 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 231.6/231.6 kB 17.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 57.5/57.5 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.7/53.7 kB 4.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.7/49.7 kB 3.9 MB/s eta 0:00:00
   ━━

In [ ]:
!pip install pyannoteai-sdk --upgrade

In [7]:
import os
import shutil
import time
import requests
import pandas as pd
from bs4 import BeautifulSoup
from urllib.parse import urlparse
from pyannoteai.sdk import Client
from google.colab import drive

drive.mount('/content/drive')

# --- CONFIGURATION ---
API_KEY = "sk_f3059e6d053b496db67d114fb13ebf71"
BASE_URL = "https://media.talkbank.org/dementia/English/Pitt/Control/cookie/"
OUTPUT_ROOT = "/content/drive/MyDrive/control/cookie/processed_audios"
TEMP_DIR = "/tmp/audio_temp"

COOKIE_STRING = "talkbank=s%3A_61nS965-sAr37VwFb0q2-qX-rxgE1CU.u82qgomB560sh6w4NdihieiWQPhZvP83LfY14zbjKRk"

session = requests.Session()
session.headers.clear()
session.headers.update({
    "Cookie": COOKIE_STRING,
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
    "Range": "bytes=0-"
})

client = Client(token=API_KEY)
os.makedirs(TEMP_DIR, exist_ok=True)

# --- STEP 0: Wipe and recreate output folder ---
print(f"🗑️  Deleting all folders in {OUTPUT_ROOT} ...")
if os.path.exists(OUTPUT_ROOT):
    shutil.rmtree(OUTPUT_ROOT)
os.makedirs(OUTPUT_ROOT)
print(f"✅ Output folder cleared and recreated.")

# --- STEP 1: Scrape .mp3 links ---
def get_audio_links(base_url):
    print(f"Fetching file list from: {base_url}")
    response = session.get(base_url, timeout=30)
    response.raise_for_status()
    print(f"  Page status: {response.status_code} | Content length: {len(response.text)}")
    soup = BeautifulSoup(response.text, "html.parser")
    parsed_base = urlparse(base_url)
    links = []
    for a in soup.find_all("a", href=True):
        href = a["href"]
        if href.lower().endswith(".mp3"):
            if href.startswith("http"):
                links.append(href)
            elif href.startswith("/"):
                links.append(f"{parsed_base.scheme}://{parsed_base.netloc}{href}")
            else:
                links.append(base_url.rstrip("/") + "/" + href)
    return sorted(links)

# --- STEP 2: Download full audio file ---
def download_audio(url, temp_dir):
    file_name = url.split("/")[-1]
    local_path = os.path.join(temp_dir, file_name)
    print(f"  Downloading: {file_name}")
    with requests.get(
        url,
        headers={
            "Cookie": COOKIE_STRING,
            "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/146.0.0.0 Safari/537.36",
            "Range": "bytes=0-"
        },
        stream=True,
        timeout=120
    ) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=65536):
                f.write(chunk)
    file_size = os.path.getsize(local_path)
    print(f"  File size: {file_size / 1024 / 1024:.2f} MB")
    if file_size < 1024:
        raise ValueError(f"Downloaded file too small ({file_size} bytes) — likely corrupt or access denied")
    return local_path, file_name

# --- STEP 3: Save RTTM helper ---
def save_rttm_file(content_or_url, target_path):
    if not content_or_url:
        return False
    try:
        if isinstance(content_or_url, str) and content_or_url.startswith("http"):
            response = requests.get(content_or_url, timeout=30)
            response.raise_for_status()
            with open(target_path, "wb") as f:
                f.write(response.content)
        else:
            with open(target_path, "w") as f:
                f.write(str(content_or_url))
        return True
    except Exception as e:
        print(f"  ⚠️  Error saving {os.path.basename(target_path)}: {e}")
        return False

# --- STEP 4: Check what's missing in a folder ---
def get_missing_files(target_folder, file_name):
    missing = []
    if not os.path.exists(os.path.join(target_folder, file_name)):
        missing.append("audio")
    if not os.path.exists(os.path.join(target_folder, "diarization.rttm")):
        missing.append("diarization")
    if not os.path.exists(os.path.join(target_folder, "exclusive_diarization.rttm")):
        missing.append("exclusive_diarization")
    if not os.path.exists(os.path.join(target_folder, "turn_level.csv")):
        missing.append("turn_level")
    return missing

# --- MAIN ---
audio_links = get_audio_links(BASE_URL)
print(f"Total .mp3 files found: {len(audio_links)}")

for audio_url in audio_links:
    file_name = audio_url.split("/")[-1]
    audio_id = os.path.splitext(file_name)[0]
    target_folder = os.path.join(OUTPUT_ROOT, audio_id)
    os.makedirs(target_folder, exist_ok=True)

    missing = get_missing_files(target_folder, file_name)

    # Case 1: Nothing missing — fully done
    if not missing:
        print(f"⏭️  Fully complete, skipping: {audio_id}")
        continue

    # Case 2: Only audio missing — just download and copy, no diarization
    if missing == ["audio"]:
        print(f"\n🔧 Audio missing, fixing: {audio_id}")
        audio_path = None
        try:
            audio_path, file_name = download_audio(audio_url, TEMP_DIR)
            shutil.copy(audio_path, os.path.join(target_folder, file_name))
            print(f"  {file_name} → copied to Drive ✅")
        except Exception as e:
            print(f"  ❌ Failed to download audio: {e}")
        finally:
            if audio_path and os.path.exists(audio_path):
                os.remove(audio_path)
                print(f"  🗑️  Temp file deleted")
        continue

    # Case 3: Diarization outputs missing — full process
    print(f"\n🚀 Processing: {audio_id} (missing: {', '.join(missing)})")

    audio_path = None
    try:
        # 1. Download full file
        audio_path, file_name = download_audio(audio_url, TEMP_DIR)

        # 2. Upload to pyannote
        uploaded_url = client.upload(audio_path)
        print(f"  Uploaded → {uploaded_url if isinstance(uploaded_url, str) else 'OK'}")

        # 3. Start diarization job
        job_id = client.diarize(
            uploaded_url,
            min_speakers=2,
            max_speakers=2,
            transcription=True,
            transcription_config={"model": "parakeet-tdt-0.6b-v3"},
            confidence=True,
            exclusive=True
        )
        print(f"  Job ID: {job_id}")

        # 4. Poll for results
        time.sleep(10)
        poll_attempts = 0

        while True:
            poll_attempts += 1
            try:
                job = client.retrieve(job_id)
            except Exception as poll_err:
                err_type = type(poll_err).__name__
                err_msg = str(poll_err)
                print(f"  ❌ [{err_type}] {err_msg}")
                if "Failed" in err_type or "failed" in err_msg.lower() or "Could not load" in err_msg:
                    print(f"  ⏭️  Skipping {audio_id} — unrecoverable job failure")
                    break
                print(f"  Retrying in 15s... (attempt {poll_attempts})")
                time.sleep(15)
                continue

            status = job.get("status")
            print(f"  [{poll_attempts}] Status: {status}")

            if status == "succeeded":
                output = job.get("output", {})

                if "diarization" in output:
                    saved = save_rttm_file(output["diarization"],
                                   os.path.join(target_folder, "diarization.rttm"))
                    print(f"  diarization.rttm → {'saved' if saved else 'FAILED'}")

                if "exclusiveDiarization" in output:
                    saved = save_rttm_file(output["exclusiveDiarization"],
                                   os.path.join(target_folder, "exclusive_diarization.rttm"))
                    print(f"  exclusive_diarization.rttm → {'saved' if saved else 'FAILED'}")

                if "turnLevelTranscription" in output:
                    data = output["turnLevelTranscription"]
                    try:
                        if isinstance(data, str) and data.startswith("http"):
                            df = pd.read_csv(data)
                        else:
                            df = pd.DataFrame(data)
                        print(f"  Transcript shape: {df.shape}")
                        df.to_csv(os.path.join(target_folder, "turn_level.csv"), index=False)
                        print(f"  turn_level.csv → saved")
                    except Exception as csv_err:
                        print(f"  ⚠️  Could not save turn_level.csv: {csv_err}")

                # Always copy audio
                shutil.copy(audio_path, os.path.join(target_folder, file_name))
                print(f"  {file_name} → copied to Drive")
                print(f"  ✅ Done: {audio_id}")
                break

            elif status == "failed":
                reason = job.get("error") or job.get("message") or "unknown"
                print(f"  ❌ Job failed: {reason}")
                break

            else:
                time.sleep(15)

    except Exception as e:
        print(f"❌ Error on {audio_id}: {e}")

    finally:
        if audio_path and os.path.exists(audio_path):
            os.remove(audio_path)
            print(f"  🗑️  Temp file deleted")

print("\n--- ALL TASKS COMPLETE ---")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
🗑️  Deleting all folders in /content/drive/MyDrive/control/cookie/processed_audios ...
✅ Output folder cleared and recreated.
Fetching file list from: https://media.talkbank.org/dementia/English/Pitt/Control/cookie/
  Page status: 200 | Content length: 74087
Total .mp3 files found: 243

🚀 Processing: 002-0 (missing: audio, diarization, exclusive_diarization, turn_level)
  Downloading: 002-0.mp3
  File size: 0.42 MB


/usr/local/lib/python3.12/dist-packages/pyannoteai/sdk/client.py:356: UserWarning: 
You are using pyannoteAI's temporary storage solution. Your file will be permanently deleted from our servers within 24hs. 
If you are running in production, we highly recommend to use your own storage to reduce network latency and obtain results faster. 
Please check our documentation at https://docs.pyannote.ai/ for more information.
  warnings.warn("""


  Uploaded → media://6dd89bd24331b9f534233e3fe2296a7c
  Job ID: 58e2f146-188b-4ebc-81f9-3362572e92fc


/usr/local/lib/python3.12/dist-packages/pyannoteai/sdk/client.py:600: UserWarning: 
You are using periodic polling to retrieve results. 
If you are running in production, we highly recommend to setup a webhook server to obtain results faster, as soon as they are available. 
Please check our documentation at https://docs.pyannote.ai/ for more information.
  warnings.warn("""


  [1] Status: succeeded
  diarization.rttm → saved
  exclusive_diarization.rttm → saved
  Transcript shape: (16, 4)
  turn_level.csv → saved
  002-0.mp3 → copied to Drive
  ✅ Done: 002-0
  🗑️  Temp file deleted

🚀 Processing: 002-1 (missing: audio, diarization, exclusive_diarization, turn_level)
  Downloading: 002-1.mp3
  File size: 0.61 MB


...

/usr/local/lib/python3.12/dist-packages/pyannoteai/sdk/client.py:600: UserWarning: 
You are using periodic polling to retrieve results. 
If you are running in production, we highly recommend to setup a webhook server to obtain results faster, as soon as they are available. 
Please check our documentation at https://docs.pyannote.ai/ for more information.
  warnings.warn("""


  [1] Status: succeeded
  diarization.rttm → saved
  exclusive_diarization.rttm → saved
  Transcript shape: (10, 4)
  turn_level.csv → saved
  709-0.mp3 → copied to Drive
  ✅ Done: 709-0
  🗑️  Temp file deleted

🚀 Processing: 709-2 (missing: audio, diarization, exclusive_diarization, turn_level)
  Downloading: 709-2.mp3
  File size: 0.28 MB


/usr/local/lib/python3.12/dist-packages/pyannoteai/sdk/client.py:356: UserWarning: 
You are using pyannoteAI's temporary storage solution. Your file will be permanently deleted from our servers within 24hs. 
If you are running in production, we highly recommend to use your own storage to reduce network latency and obtain results faster. 
Please check our documentation at https://docs.pyannote.ai/ for more information.
  warnings.warn("""


  Uploaded → media://39d5b4cccc8e530e8963859c039967cc
  Job ID: 2c44c0b2-477b-4c8d-a763-fbf7a6873c4b
  [1] Status: succeeded
  diarization.rttm → saved
  exclusive_diarization.rttm → saved
  Transcript shape: (18, 4)
  turn_level.csv → saved
  709-2.mp3 → copied to Drive
  ✅ Done: 709-2
  🗑️  Temp file deleted

--- ALL TASKS COMPLETE ---


/usr/local/lib/python3.12/dist-packages/pyannoteai/sdk/client.py:600: UserWarning: 
You are using periodic polling to retrieve results. 
If you are running in production, we highly recommend to setup a webhook server to obtain results faster, as soon as they are available. 
Please check our documentation at https://docs.pyannote.ai/ for more information.
  warnings.warn("""
